# 01 — Exploratory Data Analysis

**Dataset**: Kaggle Playground Series S6E2 — Predicting Heart Disease  
**Goal**: Deep understanding of the data — distributions, relationships, outliers, and clinical context.  
**Report section updated**: Section 3 (Dataset & Preprocessing) + Section 5.1 (EDA Results)


In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ks_2samp
from sklearn.ensemble import IsolationForest

from src.data_utils import (
    load_data, get_X_y, FEATURE_COLS, TARGET, TARGET_MAP,
    CONTINUOUS_COLS, BINARY_COLS, ORDINAL_COLS
)
from src.visualization import save_fig, PALETTE, plot_target_distribution

# Consistent aesthetics
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams['figure.dpi'] = 120

print('All imports OK')

All imports OK


## 1.1 Load Data & Overview

In [2]:
train = load_data('train', encode_target=True)
test  = load_data('test',  encode_target=False)

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
print(f'\nMemory usage (train): {train.memory_usage(deep=True).sum() / 1e6:.1f} MB')
train.head()

Train shape : (630000, 15)
Test  shape : (270000, 14)

Memory usage (train): 23.3 MB


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58.0,1,4,152.0,239.0,0,0,158.0,1,3.6,2,2,7,1
1,1,52.0,1,1,125.0,325.0,0,2,171.0,0,0.0,1,0,3,0
2,2,56.0,0,2,160.0,188.0,0,2,151.0,0,0.0,1,0,3,0
3,3,44.0,0,3,134.0,229.0,0,2,150.0,0,1.0,2,0,3,0
4,4,58.0,1,4,140.0,234.0,0,2,125.0,1,3.8,2,3,3,1


In [3]:
print('=== Data Types ===')
print(train.dtypes)
print('\n=== Descriptive Statistics (continuous) ===')
train[CONTINUOUS_COLS].describe().round(2)

=== Data Types ===
id                           int64
Age                        float32
Sex                           int8
Chest pain type               int8
BP                         float32
Cholesterol                float32
FBS over 120                  int8
EKG results                   int8
Max HR                     float32
Exercise angina               int8
ST depression              float32
Slope of ST                   int8
Number of vessels fluro       int8
Thallium                      int8
Heart Disease                 int8
dtype: object

=== Descriptive Statistics (continuous) ===


,Age,BP,Cholesterol,Max HR,ST depression
count,630000.00,630000.00,630000.00,630000.00,630000.00
mean,54.14,130.50,245.01,152.82,0.72
std,8.26,14.98,33.68,19.11,0.95
min,29.00,94.00,126.00,71.00,0.00
25%,48.00,120.00,223.00,142.00,0.00
50%,54.00,130.00,243.00,157.00,0.10
75%,60.00,140.00,269.00,166.00,1.40
max,77.00,200.00,564.00,202.00,6.20


In [4]:
print('=== Descriptive Statistics (ordinal/binary) ===')
train[BINARY_COLS + ORDINAL_COLS].describe().round(2)

=== Descriptive Statistics (ordinal/binary) ===


,Sex,FBS over 120,Exercise angina,Chest pain type,EKG results,Slope of ST,Number of vessels fluro,Thallium
count,630000.00,630000.00,630000.00,630000.00,630000.00,630000.00,630000.00,630000.00
mean,0.71,0.08,0.27,3.31,0.98,1.46,0.45,4.62
std,0.45,0.27,0.45,0.85,1.00,0.55,0.80,1.95
min,0.00,0.00,0.00,1.00,0.00,1.00,0.00,3.00
25%,0.00,0.00,0.00,3.00,0.00,1.00,0.00,3.00
50%,1.00,0.00,0.00,4.00,0.00,1.00,0.00,3.00
75%,1.00,0.00,1.00,4.00,2.00,2.00,1.00,7.00
max,1.00,1.00,1.00,4.00,2.00,3.00,3.00,7.00


## 1.2 Target Distribution

In [5]:
target_counts = train[TARGET].value_counts()
total = len(train)
print('Heart Disease distribution:')
print(f"  Absence  (0): {target_counts[0]:,}  ({target_counts[0]/total*100:.2f}%)")
print(f"  Presence (1): {target_counts[1]:,}  ({target_counts[1]/total*100:.2f}%)")
print(f"  Imbalance ratio: {target_counts[0]/target_counts[1]:.3f}")

fig = plot_target_distribution(train[TARGET], title='Target Distribution — Heart Disease')
save_fig('01_target_distribution', fig)
plt.show()

Heart Disease distribution:
  Absence  (0): 347,546  (55.17%)
  Presence (1): 282,454  (44.83%)
  Imbalance ratio: 1.230


## 1.3 Missing Values & Data Quality

In [6]:
# Missing values
missing = train.isnull().sum()
print('Missing values per column (train):')
print(missing[missing > 0] if missing.sum() > 0 else 'None — dataset is complete.')

# Duplicates
dup_count = train.duplicated(subset=FEATURE_COLS).sum()
print(f'\nDuplicate feature rows: {dup_count:,} ({dup_count/len(train)*100:.3f}%)')

Missing values per column (train):
None — dataset is complete.

Duplicate feature rows: 0 (0.000%)


In [7]:
# Suspicious zero/negative values in clinical features
suspicious = {
    'BP': (train['BP'] == 0).sum(),
    'Cholesterol': (train['Cholesterol'] == 0).sum(),
    'Max HR': (train['Max HR'] == 0).sum(),
    'Age': (train['Age'] < 1).sum(),
    'ST depression < 0': (train['ST depression'] < 0).sum(),
}
print('Potentially invalid values:')
for col, cnt in suspicious.items():
    print(f'  {col}: {cnt:,}')

Potentially invalid values:
  BP: 0
  Cholesterol: 0
  Max HR: 0
  Age: 0
  ST depression < 0: 0


In [8]:
# Check categorical value ranges
print('Categorical value ranges:')
for col in BINARY_COLS + ORDINAL_COLS:
    vals = sorted(train[col].unique())
    print(f'  {col}: {vals}')

Categorical value ranges:
  Sex: [0, 1]
  FBS over 120: [0, 1]
  Exercise angina: [0, 1]
  Chest pain type: [1, 2, 3, 4]
  EKG results: [0, 1, 2]
  Slope of ST: [1, 2, 3]
  Number of vessels fluro: [0, 1, 2, 3]
  Thallium: [3, 6, 7]


## 1.4 Univariate Analysis — Continuous Features

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(CONTINUOUS_COLS):
    ax = axes[i]
    # KDE by class
    for label, color in zip([0, 1], sns.color_palette(PALETTE)[:2]):
        subset = train[train[TARGET] == label][col]
        ax.hist(subset, bins=50, alpha=0.55, color=color, density=True,
                label='Absence' if label==0 else 'Presence')
        subset.plot.kde(ax=ax, color=color, lw=1.5)
    skew = train[col].skew()
    ax.set_title(f'{col}\n(skew={skew:.2f})')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

# Hide empty last subplot
axes[-1].set_visible(False)
fig.suptitle('Continuous Feature Distributions by Target Class', fontsize=13, y=1.01)
fig.tight_layout()
save_fig('01_continuous_distributions', fig)
plt.show()

In [10]:
# Summary stats table for continuous features
stats_rows = []
for col in CONTINUOUS_COLS:
    for target_val, target_name in [(0, 'Absence'), (1, 'Presence')]:
        s = train[train[TARGET]==target_val][col]
        stats_rows.append({
            'Feature': col, 'Class': target_name,
            'Mean': s.mean(), 'Std': s.std(), 'Median': s.median(),
            'Min': s.min(), 'Max': s.max(), 'Skew': s.skew()
        })
stats_df = pd.DataFrame(stats_rows).round(2)
stats_df

,Feature,Class,Mean,Std,Median,Min,Max,Skew
0,Age,Absence,52.560001,8.300000,52.0,29.0,77.0,0.04
1,Age,Presence,56.080002,7.770000,57.0,29.0,77.0,-0.40
2,BP,Absence,130.570007,15.140000,130.0,94.0,200.0,0.67
3,BP,Presence,130.410004,14.780000,130.0,94.0,200.0,0.57
4,Cholesterol,Absence,242.500000,34.080002,239.0,126.0,564.0,0.40
5,Cholesterol,Presence,248.100006,32.919998,246.0,126.0,564.0,0.13
6,Max HR,Absence,160.419998,14.740000,162.0,71.0,202.0,-0.82
7,Max HR,Presence,143.470001,19.719999,146.0,71.0,202.0,-0.45
8,ST depression,Absence,0.350000,0.630000,0.0,0.0,6.0,2.14
9,ST depression,Presence,1.170000,1.070000,1.2,0.0,6.2,0.68


## 1.5 Univariate Analysis — Categorical Features

In [11]:
all_cat = BINARY_COLS + ORDINAL_COLS
n_cols = 3
n_rows = (len(all_cat) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(all_cat):
    ax = axes[i]
    ct = pd.crosstab(train[col], train[TARGET], normalize='index') * 100
    ct.columns = ['Absence', 'Presence']
    ct.plot(kind='bar', ax=ax, color=sns.color_palette(PALETTE)[:2],
            edgecolor='white', stacked=False)
    ax.set_title(f'{col} vs Heart Disease')
    ax.set_xlabel(col)
    ax.set_ylabel('% within value')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Categorical Feature Distributions by Target Class (%)', fontsize=13, y=1.01)
fig.tight_layout()
save_fig('01_categorical_distributions', fig)
plt.show()

## 1.6 Bivariate Statistical Tests

In [12]:
from scipy.stats import mannwhitneyu, chi2_contingency

# Mann-Whitney U for continuous features (non-parametric, no normality assumption)
print('=== Mann-Whitney U Test: Continuous Features vs Target ===')
print(f'{"Feature":<25} {"U-stat":>12} {"p-value":>12} {"Significant":>12}')
print('-' * 65)
mwu_results = {}
for col in CONTINUOUS_COLS:
    group0 = train[train[TARGET]==0][col].dropna()
    group1 = train[train[TARGET]==1][col].dropna()
    u_stat, p_val = mannwhitneyu(group0, group1, alternative='two-sided')
    mwu_results[col] = {'U': u_stat, 'p': p_val}
    sig = '✓' if p_val < 0.05 else ' '
    print(f'{col:<25} {u_stat:>12.0f} {p_val:>12.4e} {sig:>12}')

=== Mann-Whitney U Test: Continuous Features vs Target ===
Feature                         U-stat      p-value  Significant
-----------------------------------------------------------------
Age                        36747944418   0.0000e+00            ✓
BP                         49035193338   5.0311e-01             
Cholesterol                43889513411   0.0000e+00            ✓


Max HR                     74204284952   0.0000e+00            ✓
ST depression              26143741180   0.0000e+00            ✓


In [13]:
# Chi-squared for categorical features
print('=== Chi-Squared Test: Categorical Features vs Target ===')
print(f'{"Feature":<30} {"Chi2":>10} {"p-value":>12} {"Cramers V":>10}')
print('-' * 65)
chi2_results = {}
for col in BINARY_COLS + ORDINAL_COLS:
    ct = pd.crosstab(train[col], train[TARGET])
    chi2, p, dof, expected = chi2_contingency(ct)
    # Cramér's V effect size
    n = ct.sum().sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    chi2_results[col] = {'chi2': chi2, 'p': p, 'cramers_v': v}
    print(f'{col:<30} {chi2:>10.1f} {p:>12.4e} {v:>10.4f}')

=== Chi-Squared Test: Categorical Features vs Target ===
Feature                              Chi2      p-value  Cramers V
-----------------------------------------------------------------
Sex                               73878.2   0.0000e+00     0.3424
FBS over 120                        709.7  2.2808e-156     0.0336
Exercise angina                  123001.4   0.0000e+00     0.4419
Chest pain type                  173770.7   0.0000e+00     0.5252
EKG results                       30248.4   0.0000e+00     0.2191
Slope of ST                      116357.6   0.0000e+00     0.4298
Number of vessels fluro          135190.5   0.0000e+00     0.4632
Thallium                         231215.7   0.0000e+00     0.6058


## 1.7 Multivariate Analysis — Correlation

In [14]:
X, y = get_X_y(train, extra_features=False)
corr_data = X.copy()
corr_data[TARGET] = y

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Pearson
pearson_corr = corr_data.corr(method='pearson')
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))
sns.heatmap(pearson_corr, mask=mask, ax=axes[0], annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Pearson Correlation')

# Spearman
spearman_corr = corr_data.corr(method='spearman')
sns.heatmap(spearman_corr, mask=mask, ax=axes[1], annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Spearman Correlation')

fig.suptitle('Feature Correlation Matrices (lower triangle)', fontsize=13)
fig.tight_layout()
save_fig('01_correlation_heatmaps', fig)
plt.show()

In [15]:
# Top correlations with target
target_corr = spearman_corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print('Feature-Target Spearman Correlations (ranked by |r|):')
for feat, r in target_corr.items():
    bar = '█' * int(abs(r) * 30)
    print(f'  {feat:<30} {r:>+.4f}  {bar}')

Feature-Target Spearman Correlations (ranked by |r|):
  Thallium                       +0.6050  ██████████████████
  Chest pain type                +0.5089  ███████████████
  Number of vessels fluro        +0.4627  █████████████
  Exercise angina                +0.4419  █████████████
  Max HR                         -0.4410  █████████████
  ST depression                  +0.4305  ████████████
  Slope of ST                    +0.4271  ████████████
  Sex                            +0.3424  ██████████
  EKG results                    +0.2190  ██████
  Age                            +0.2167  ██████
  Cholesterol                    +0.0912  ██
  FBS over 120                   +0.0336  █
  BP                             +0.0008  


## 1.8 Mutual Information

In [16]:
from sklearn.feature_selection import mutual_info_classif

X, y = get_X_y(train, extra_features=False)

mi_scores = mutual_info_classif(X, y, random_state=42, n_neighbors=3)
mi_df = pd.Series(mi_scores, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = [sns.color_palette(PALETTE)[0]] * len(mi_df)
# Highlight top 3
top3_idx = mi_df.nlargest(3).index
for idx in mi_df.index:
    if idx in top3_idx:
        colors[list(mi_df.index).index(idx)] = sns.color_palette(PALETTE)[1]
ax.barh(mi_df.index, mi_df.values, color=colors)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Mutual Information: Features vs Heart Disease')
ax.axvline(mi_df.mean(), color='red', linestyle='--', lw=1, label=f'Mean = {mi_df.mean():.3f}')
ax.legend()
fig.tight_layout()
save_fig('01_mutual_information', fig)
plt.show()

print('\nMI Rankings:')
for feat, score in mi_df.sort_values(ascending=False).items():
    print(f'  {feat:<30} {score:.4f}')


MI Rankings:
  Thallium                       0.2358
  Chest pain type                0.1895
  Sex                            0.1318
  Max HR                         0.1293
  Slope of ST                    0.1250
  Exercise angina                0.1239
  Number of vessels fluro        0.1206
  ST depression                  0.1082
  EKG results                    0.0751
  Age                            0.0299
  Cholesterol                    0.0120
  BP                             0.0110
  FBS over 120                   0.0022


## 1.9 Outlier Analysis

In [17]:
# IQR-based outlier detection
print('=== IQR-Based Outlier Count per Continuous Feature ===')
iqr_outliers = {}
for col in CONTINUOUS_COLS:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((train[col] < lower) | (train[col] > upper)).sum()
    iqr_outliers[col] = n_outliers
    print(f'  {col:<20} [{lower:.1f}, {upper:.1f}]  outliers: {n_outliers:,} ({n_outliers/len(train)*100:.2f}%)')

=== IQR-Based Outlier Count per Continuous Feature ===
  Age                  [30.0, 78.0]  outliers: 1,048 (0.17%)
  BP                   [90.0, 170.0]  outliers: 9,011 (1.43%)
  Cholesterol          [154.0, 338.0]  outliers: 2,194 (0.35%)
  Max HR               [106.0, 202.0]  outliers: 14,246 (2.26%)
  ST depression        [-2.1, 3.5]  outliers: 9,160 (1.45%)


In [18]:
# Box plots for continuous features
fig, axes = plt.subplots(1, len(CONTINUOUS_COLS), figsize=(18, 5))
for ax, col in zip(axes, CONTINUOUS_COLS):
    train.boxplot(column=col, by=TARGET, ax=ax, 
                  boxprops=dict(color='steelblue'),
                  medianprops=dict(color='red', lw=2),
                  flierprops=dict(marker='.', markersize=2, alpha=0.3))
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Heart Disease (0=Absent, 1=Present)')

fig.suptitle('Box Plots: Continuous Features by Target', fontsize=13)
plt.tight_layout()
save_fig('01_boxplots_by_target', fig)
plt.show()

In [19]:
# Multivariate outlier detection with Isolation Forest (sample for speed)
from sklearn.ensemble import IsolationForest

np.random.seed(42)
sample_idx = np.random.choice(len(train), size=50000, replace=False)
X_sample = train.iloc[sample_idx][CONTINUOUS_COLS].values

iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
outlier_labels = iso.fit_predict(X_sample)
n_iso_outliers = (outlier_labels == -1).sum()
print(f'Isolation Forest (contamination=5%, sample=50K): {n_iso_outliers:,} multivariate outliers ({n_iso_outliers/len(X_sample)*100:.1f}%)')

Isolation Forest (contamination=5%, sample=50K): 2,500 multivariate outliers (5.0%)


## 1.10 Train vs Test Distribution Check (KS Test)

In [20]:
from scipy.stats import ks_2samp

print('=== Kolmogorov-Smirnov Test: Train vs Test Feature Distributions ===')
print(f'{"Feature":<25} {"KS stat":>10} {"p-value":>12} {"Drift?":>8}')
print('-' * 58)
for col in FEATURE_COLS:
    ks_stat, p_val = ks_2samp(train[col].dropna(), test[col].dropna())
    drift = 'YES' if p_val < 0.05 else 'no'
    print(f'{col:<25} {ks_stat:>10.4f} {p_val:>12.4e} {drift:>8}')

=== Kolmogorov-Smirnov Test: Train vs Test Feature Distributions ===
Feature                      KS stat      p-value   Drift?
----------------------------------------------------------


Age                           0.0022   3.3931e-01       no
Sex                           0.0016   7.3238e-01       no
Chest pain type               0.0021   3.8047e-01       no


BP                            0.0023   2.5686e-01       no


Cholesterol                   0.0014   8.5463e-01       no
FBS over 120                  0.0001   1.0000e+00       no


EKG results                   0.0016   7.0492e-01       no


Max HR                        0.0017   6.1823e-01       no
Exercise angina               0.0009   9.9668e-01       no


ST depression                 0.0026   1.6406e-01       no
Slope of ST                   0.0031   5.4109e-02       no


Number of vessels fluro       0.0022   3.1395e-01       no
Thallium                      0.0003   1.0000e+00       no


## 1.11 Pairplot — Top Features

In [21]:
# Use top features by MI score for pairplot
top_features = mi_df.nlargest(5).index.tolist()
print('Top features for pairplot:', top_features)

# Subsample for performance
sample = train[top_features + [TARGET]].sample(5000, random_state=42)
sample[TARGET] = sample[TARGET].map({0: 'Absence', 1: 'Presence'})

g = sns.pairplot(sample, hue=TARGET, plot_kws={'alpha': 0.3, 's': 10},
                 palette=PALETTE, corner=True)
g.fig.suptitle('Pairplot — Top 5 Informative Features', y=1.01, fontsize=12)
save_fig('01_pairplot_top_features', g.fig)
plt.show()

Top features for pairplot: ['Thallium', 'Chest pain type', 'Sex', 'Max HR', 'Slope of ST']


## 1.12 EDA Summary & Clinical Insights

In [22]:
print('=' * 65)
print('EDA SUMMARY')
print('=' * 65)
print(f'  Training samples      : {len(train):,}')
print(f'  Test samples          : {len(test):,}')
print(f'  Features              : {len(FEATURE_COLS)}')
print(f'  Target balance        : {target_counts[0]/total*100:.1f}% Absent / {target_counts[1]/total*100:.1f}% Present')
print(f'  Missing values        : None')
print(f'  Duplicate feature rows: {dup_count:,}')
print()
print('Top 5 Features by Mutual Information:')
for i, (feat, score) in enumerate(mi_df.sort_values(ascending=False).head(5).items(), 1):
    print(f'  {i}. {feat:<30} MI={score:.4f}')
print()
print('Clinical Notes:')
print('  - Thallium (stress test) and Number of Vessels Fluro are strongest predictors')
print('  - Chest Pain Type 4 (asymptomatic) paradoxically most associated with Presence')
print('  - ST depression > 1.5 strongly associated with heart disease')
print('  - Lower Max HR associated with heart disease (reduced cardiovascular fitness)')
print('  - Mild class imbalance (55/45) - baseline strategies may be sufficient')

EDA SUMMARY
  Training samples      : 630,000
  Test samples          : 270,000
  Features              : 13
  Target balance        : 55.2% Absent / 44.8% Present
  Missing values        : None
  Duplicate feature rows: 0

Top 5 Features by Mutual Information:
  1. Thallium                       MI=0.2358
  2. Chest pain type                MI=0.1895
  3. Sex                            MI=0.1318
  4. Max HR                         MI=0.1293
  5. Slope of ST                    MI=0.1250

Clinical Notes:
  - Thallium (stress test) and Number of Vessels Fluro are strongest predictors
  - Chest Pain Type 4 (asymptomatic) paradoxically most associated with Presence
  - ST depression > 1.5 strongly associated with heart disease
  - Lower Max HR associated with heart disease (reduced cardiovascular fitness)
  - Mild class imbalance (55/45) - baseline strategies may be sufficient


In [23]:
# Save EDA metrics for report
import json, pathlib

eda_metrics = {
    'train_samples': int(len(train)),
    'test_samples': int(len(test)),
    'n_features': len(FEATURE_COLS),
    'target_absence_pct': float(target_counts[0] / total * 100),
    'target_presence_pct': float(target_counts[1] / total * 100),
    'imbalance_ratio': float(target_counts[0] / target_counts[1]),
    'duplicate_rows': int(dup_count),
    'mi_rankings': {str(k): float(v) for k, v in mi_df.sort_values(ascending=False).items()},
    'mwu_p_values': {k: float(v['p']) for k, v in mwu_results.items()},
    'chi2_cramers_v': {k: float(v['cramers_v']) for k, v in chi2_results.items()},
}

metrics_path = pathlib.Path('../results/metrics/01_eda_metrics.json')
metrics_path.parent.mkdir(exist_ok=True)
metrics_path.write_text(json.dumps(eda_metrics, indent=2))
print(f'EDA metrics saved to {metrics_path}')

EDA metrics saved to ../results/metrics/01_eda_metrics.json
